In [4]:
import pandas as pd
import numpy as np
import scipy as scp
import dgl 
import math
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
ROOT_PATH = '/Users/keltonn/Desktop/Projects/Data/location_data_mining_cs5228/00_location_graph_data_mining'

# Train Stations

In [9]:
mrt_node = pd.read_csv(f'{ROOT_PATH}/00_location_graph_data_mining/raw_graph_data/sg-mrt-nodes.csv')
mrt_node

,NAME,member_codes,lines,n_lines,status,latitude,longitude,subzone,planning_area,region,is_interchange
0,admiralty mrt station,ns10,ns,1,open,1.440689,103.800933,woodlands east,woodlands,north region,False
1,aljunied mrt station,ew9,ew,1,open,1.316389,103.882778,aljunied,geylang,central region,False
2,ang mo kio mrt station,cr11;ns16,cr;ns,2,open,1.370017,103.849450,cheng san,ang mo kio,north-east region,True
3,aviation park mrt station,cr2,cr,1,open,1.370000,104.003056,changi airport,changi,east region,False
4,bahar junction mrt station,js7,js,1,open,1.345422,103.702542,jurong west central,jurong west,west region,False
...,...,...,...,...,...,...,...,...,...,...,...
187,woodleigh mrt station,ne11,ne,1,open,1.339167,103.870833,bidadari,toa payoh,central region,False
188,xilin mrt station,dt36,dt,1,open,1.328889,103.965000,xilin,tampines,east region,False
189,yew tee mrt station,ns5,ns,1,open,1.397500,103.747500,yew tee,choa chu kang,west region,False
190,yio chu kang mrt station,ns15,ns,1,open,1.381906,103.844817,yio chu kang,ang mo kio,north-east region,False


In [11]:
mall_feature = pd.read_csv(f'{ROOT_PATH}/data/collected_data/mrt_data/sg-station-mall-counts.csv')
mall_feature.columns = ['NAME','n_malls_500m','nearest_mall','nearest_mall_m','malls_500m']
mall_feature = mall_feature[['NAME','n_malls_500m','malls_500m','nearest_mall','nearest_mall_m',]]
mall_feature

,NAME,n_malls_500m,malls_500m,nearest_mall,nearest_mall_m
0,admiralty mrt station,0,NaN,Causeway Point,1739.6
1,aljunied mrt station,0,NaN,Lot 1,979.0
2,ang mo kio mrt station,1,AMK Hub,AMK Hub,128.6
3,aviation park mrt station,0,NaN,Jewel Changi Airport,1889.0
4,bahar junction mrt station,0,NaN,Gek Poh Shopping Centre,649.9
...,...,...,...,...,...
187,woodleigh mrt station,0,NaN,NEX,1300.3
188,xilin mrt station,0,NaN,Changi City Point,617.4
189,yew tee mrt station,1,Yew Tee Point,Yew Tee Point,68.8
190,yio chu kang mrt station,0,NaN,AMK Hub,1450.1


In [14]:
df = pd.merge(mrt_node, mall_feature, how = 'left', on = 'NAME')
df

,NAME,member_codes,lines,n_lines,status,latitude,longitude,subzone,planning_area,region,is_interchange,n_malls_500m,malls_500m,nearest_mall,nearest_mall_m
0,admiralty mrt station,ns10,ns,1,open,1.440689,103.800933,woodlands east,woodlands,north region,False,0,NaN,Causeway Point,1739.6
1,aljunied mrt station,ew9,ew,1,open,1.316389,103.882778,aljunied,geylang,central region,False,0,NaN,Lot 1,979.0
2,ang mo kio mrt station,cr11;ns16,cr;ns,2,open,1.370017,103.849450,cheng san,ang mo kio,north-east region,True,1,AMK Hub,AMK Hub,128.6
3,aviation park mrt station,cr2,cr,1,open,1.370000,104.003056,changi airport,changi,east region,False,0,NaN,Jewel Changi Airport,1889.0
4,bahar junction mrt station,js7,js,1,open,1.345422,103.702542,jurong west central,jurong west,west region,False,0,NaN,Gek Poh Shopping Centre,649.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,woodleigh mrt station,ne11,ne,1,open,1.339167,103.870833,bidadari,toa payoh,central region,False,0,NaN,NEX,1300.3
188,xilin mrt station,dt36,dt,1,open,1.328889,103.965000,xilin,tampines,east region,False,0,NaN,Changi City Point,617.4
189,yew tee mrt station,ns5,ns,1,open,1.397500,103.747500,yew tee,choa chu kang,west region,False,1,Yew Tee Point,Yew Tee Point,68.8
190,yio chu kang mrt station,ns15,ns,1,open,1.381906,103.844817,yio chu kang,ang mo kio,north-east region,False,0,NaN,AMK Hub,1450.1


In [15]:
df.to_csv(f'{ROOT_PATH}/00_location_graph_data_mining/graph_feature_store/station-keys.csv', index = False)

# Train Traffic

In [16]:
dates = ['202605','202606','202607']
traffic_lst = []
for d in dates:
    traffic = pd.read_csv(f'{ROOT_PATH}/data/collected_data/mrt_data/sg-mrt-nodes-features-{d}.csv')
    traffic = traffic[traffic['pv_matched'] == True].reset_index(drop = True)
    traffic = traffic[['NAME',
        'pv_tap_in', 'pv_tap_out', 'pv_total', 'pv_weekday_total',
        'pv_weekend_total', 'pv_am_peak_total', 'pv_pm_peak_total',
        'pv_busiest_hour', 'year_month',]]
    traffic_lst.append(traffic)

traffic_df = pd.concat(traffic_lst).reset_index(drop = True)
traffic_df


,NAME,pv_tap_in,pv_tap_out,pv_total,pv_weekday_total,pv_weekend_total,pv_am_peak_total,pv_pm_peak_total,pv_busiest_hour,year_month
0,admiralty mrt station,1220591.0,1088699.0,2309290.0,1601448.0,707842.0,447168.0,423421.0,7.0,2026-05
1,aljunied mrt station,701918.0,708113.0,1410031.0,874144.0,535887.0,210393.0,239064.0,18.0,2026-05
2,ang mo kio mrt station,1220599.0,1339014.0,2559613.0,1726662.0,832951.0,408605.0,466877.0,18.0,2026-05
3,bartley mrt station,254624.0,228521.0,483145.0,353537.0,129608.0,101289.0,86506.0,7.0,2026-05
4,bayfront mrt station,1024518.0,966289.0,1990807.0,1046697.0,944110.0,133522.0,254453.0,20.0,2026-05
...,...,...,...,...,...,...,...,...,...,...
427,woodlands south mrt station,402335.0,373959.0,776294.0,637179.0,139115.0,176196.0,164702.0,7.0,2026-07
428,woodleigh mrt station,467808.0,480973.0,948781.0,741200.0,207581.0,167851.0,211816.0,18.0,2026-07
429,yew tee mrt station,919637.0,886547.0,1806184.0,1434612.0,371572.0,376052.0,383765.0,7.0,2026-07
430,yio chu kang mrt station,687769.0,640383.0,1328152.0,1157293.0,170859.0,346223.0,308066.0,8.0,2026-07


In [17]:
traffic_df.to_csv(f'{ROOT_PATH}/00_location_graph_data_mining/graph_feature_store/station-traffic.csv', index= False)

# Traffic Aggregation

In [195]:
num_col = ['pv_tap_in', 'pv_tap_out', 'pv_total', 'pv_weekday_total',
       'pv_weekend_total', 'pv_am_peak_total', 'pv_pm_peak_total',
       ]
cat_col = ['pv_busiest_hour',]


traffic_df = pd.read_csv(f'{ROOT_PATH}/00_location_graph_data_mining/graph_feature_store/station-traffic.csv')
traffic_df = traffic_df[traffic_df['year_month'] == '2026-07']

traffic_df

,NAME,pv_tap_in,pv_tap_out,pv_total,pv_weekday_total,pv_weekend_total,pv_am_peak_total,pv_pm_peak_total,pv_busiest_hour,year_month
288,admiralty mrt station,1297451.0,1148371.0,2445822.0,1948912.0,496910.0,545417.0,515983.0,7.0,2026-07
289,aljunied mrt station,725638.0,733965.0,1459603.0,1077763.0,381840.0,255792.0,296343.0,18.0,2026-07
290,ang mo kio mrt station,1274394.0,1408757.0,2683151.0,2104497.0,578654.0,499423.0,571260.0,18.0,2026-07
291,bartley mrt station,288510.0,255471.0,543981.0,443414.0,100567.0,124610.0,110367.0,7.0,2026-07
292,bayfront mrt station,1140191.0,1055521.0,2195712.0,1469332.0,726380.0,166977.0,354622.0,20.0,2026-07
...,...,...,...,...,...,...,...,...,...,...
427,woodlands south mrt station,402335.0,373959.0,776294.0,637179.0,139115.0,176196.0,164702.0,7.0,2026-07
428,woodleigh mrt station,467808.0,480973.0,948781.0,741200.0,207581.0,167851.0,211816.0,18.0,2026-07
429,yew tee mrt station,919637.0,886547.0,1806184.0,1434612.0,371572.0,376052.0,383765.0,7.0,2026-07
430,yio chu kang mrt station,687769.0,640383.0,1328152.0,1157293.0,170859.0,346223.0,308066.0,8.0,2026-07


In [ ]:
scale = StandardScaler()
tdf = traffic_df.copy(deep = True)
tdf[num_col] = scale.fit_transform(traffic_df[num_col])
joblib.dump(scale, '/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/scalers/traffic_scale.pkl')
tdf = tdf.drop('year_month', axis = 1)
tdf

,NAME,pv_tap_in,pv_tap_out,pv_total,pv_weekday_total,pv_weekend_total,pv_am_peak_total,pv_pm_peak_total,pv_busiest_hour
288,admiralty mrt station,1.397545,1.058514,1.227654,1.291102,0.943364,1.782587,1.098732,7.0
289,aljunied mrt station,0.229524,0.239906,0.235274,0.154663,0.472060,0.137620,0.115453,18.0
290,ang mo kio mrt station,1.350448,1.572875,1.466466,1.494068,1.278170,1.521358,1.346194,18.0
291,bartley mrt station,-0.663381,-0.705300,-0.686068,-0.672864,-0.679976,-0.607448,-0.717120,7.0
292,bayfront mrt station,1.076316,0.875100,0.975982,0.665476,1.883226,-0.366818,0.376355,20.0
...,...,...,...,...,...,...,...,...,...
427,woodlands south mrt station,-0.430875,-0.471241,-0.452304,-0.420092,-0.522092,-0.314457,-0.473874,7.0
428,woodleigh mrt station,-0.297136,-0.259848,-0.278739,-0.284393,-0.241669,-0.361854,-0.262955,18.0
429,yew tee mrt station,0.625798,0.541313,0.584020,0.620183,0.430005,0.820654,0.506821,7.0
430,yio chu kang mrt station,0.152170,0.055046,0.103002,0.258412,-0.392075,0.651236,0.167934,8.0


In [199]:
tdf = traffic_df.copy(deep = True).drop('year_month', axis = 1)
tdf.columns = ['NAME', 'mrt_traffic_tap_in_amount_norm', 'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm', 'mrt_traffic_weekday_tap_norm',
       'mrt_traffic_weekend_tap_norm', 'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm',
       'mrt_traffic_busiest_hour']

In [200]:
tdf.to_csv('/Users/keltonn/Desktop/Repo/CS5228-project/00_location_graph_data_mining/graph_feature_store/station-traffic-std.csv', index = False)

# Get Nearest Stations for each house

In [2]:
LAT0 = 1.35
R_EARTH = 6371000.0
def to_xy(lat, lon):
    x = np.radians(np.asarray(lon, float)) * math.cos(math.radians(LAT0)) * R_EARTH
    y = np.radians(np.asarray(lat, float)) * R_EARTH
    return np.column_stack([np.ravel(x), np.ravel(y)])

In [5]:
stations = pd.read_csv(f'{ROOT_PATH}/graph_feature_store/station-keys.csv')[['NAME','latitude','longitude']]
houses = pd.read_csv(f'{ROOT_PATH}/raw_graph_data/sg-hdb-block.csv')[['TOWN','BLOCK','STREET','POSTAL_CODE','LATITUDE','LONGITUDE']]
houses

,TOWN,BLOCK,STREET,POSTAL_CODE,LATITUDE,LONGITUDE
0,serangoon,106,serangoon north avenue 1,550106,1.370245,103.870823
1,bukit merah,34,telok blangah way,090034,1.273518,103.822481
2,punggol,226b,sumang lane,822226,1.402685,103.893609
3,marine parade,74,marine drive,440074,1.303705,103.907214
4,choa chu kang,284,choa chu kang avenue 3,NIL,1.379714,103.742138
...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,310155,1.332796,103.845589
9460,sembawang,307,canberra road,750307,1.445923,103.821657
9461,woodlands,681a,woodlands drive 62,731681,1.439534,103.803797
9462,choa chu kang,673a,choa chu kang crescent,681673,1.401014,103.747524


In [ ]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(stations.latitude.values, stations.longitude.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

link_rows, count_rows = [], []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_station": stations.NAME[nearest],
        "nearest_station_m": round(float(d[nearest]), 1),
    }

    for WALK_M in [500,1000]:
        within = np.where(d <= WALK_M)[0]
        order = within[np.argsort(d[within])]  # nearest first
        for rank, m in enumerate(order, start=1):
            link_rows.append({
                "house_town": houses.TOWN[s], "house_block": houses.BLOCK[s], "house_street": houses.STREET[s], "station_name": stations.NAME[m],
                "distance_m": round(float(d[m]), 1), "rank": rank,
            })
        results[f"n_stations_{int(WALK_M)}m"] = len(order)
        # results[f"stations_{int(WALK_M)}m"]=  ";".join(stations.NAME[order].tolist())

    count_rows.append(results)

links = pd.DataFrame(link_rows).sort_values(["station_name", "rank"])
counts = pd.DataFrame(count_rows)

In [8]:
counts.to_csv(f'{ROOT_PATH}/graph_feature_store/hdb_nearest_station.csv', index = False)

In [58]:
temp = pd.read_csv('/Users/keltonn/Desktop/Projects/Data/location_data_mining_cs5228/99_predictions/data/train_feature.csv')
temp[['TOWN','BLOCK','STREET']].drop_duplicates()

,TOWN,BLOCK,STREET
0,punggol,604a,punggol road
1,sengkang,290c,compassvale crescent
2,queenstown,5,ghim moh road
3,sengkang,436b,fernvale road
4,sembawang,467b,admiralty drive
...,...,...,...
148047,yishun,479c,yishun street 42
148051,bukit batok,463b,bukit batok street 41
148430,queenstown,41,stirling road
148863,choa chu kang,802b,keat hong close


# Get Nearest Schools

In [84]:
schools = pd.read_csv(f'{ROOT_PATH}/raw_model_data/auxiliary/sg-schools.csv')[['NAME','TYPE_CODE','NATRUE_CODE','MAINLEVEL_CODE','LATITUDE','LONGITUDE']]
houses = pd.read_csv(f'{ROOT_PATH}/raw_graph_data/sg-hdb-block.csv')[['TOWN','BLOCK','STREET','POSTAL_CODE','LATITUDE','LONGITUDE']]
schools.columns = ['nearest_school','nearest_school_type','nearest_school_nature','nearest_school_level','LATITUDE','LONGITUDE']
sch_type = schools[['nearest_school','nearest_school_type','nearest_school_nature']]
display(schools)
display(houses)

,nearest_school,nearest_school_type,nearest_school_nature,nearest_school_level,LATITUDE,LONGITUDE
0,admiralty primary school,government school,co-ed school,primary,1.442635,103.800040
1,admiralty secondary school,government school,co-ed school,secondary (s1-s5),1.445891,103.802398
2,ahmad ibrahim primary school,government school,co-ed school,primary,1.433153,103.832942
3,ahmad ibrahim secondary school,government school,co-ed school,secondary (s1-s5),1.436060,103.829719
4,ai tong school,government-aided sch,co-ed school,primary,1.360583,103.833020
...,...,...,...,...,...,...
332,zhangde primary school,government school,co-ed school,primary,1.284212,103.825952
333,zhenghua primary school,government school,co-ed school,primary,1.379549,103.769314
334,zhenghua secondary school,government school,co-ed school,secondary (s1-s5),1.388366,103.765511
335,zhonghua primary school,government school,co-ed school,primary,1.360203,103.869762


,TOWN,BLOCK,STREET,POSTAL_CODE,LATITUDE,LONGITUDE
0,serangoon,106,serangoon north avenue 1,550106,1.370245,103.870823
1,bukit merah,34,telok blangah way,090034,1.273518,103.822481
2,punggol,226b,sumang lane,822226,1.402685,103.893609
3,marine parade,74,marine drive,440074,1.303705,103.907214
4,choa chu kang,284,choa chu kang avenue 3,NIL,1.379714,103.742138
...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,310155,1.332796,103.845589
9460,sembawang,307,canberra road,750307,1.445923,103.821657
9461,woodlands,681a,woodlands drive 62,731681,1.439534,103.803797
9462,choa chu kang,673a,choa chu kang crescent,681673,1.401014,103.747524


In [85]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(schools.LATITUDE.values, schools.LONGITUDE.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

count_rows = []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_school": schools.nearest_school[nearest],
        "nearest_school_m": round(float(d[nearest]), 1),
    }

    count_rows.append(results)

counts = pd.DataFrame(count_rows)
counts = pd.merge(counts, schools[['nearest_school', 'nearest_school_type', 'nearest_school_nature',
       'nearest_school_level',]], how = 'left', on = 'nearest_school')
counts

,TOWN,BLOCK,STREET,nearest_school,nearest_school_m,nearest_school_type,nearest_school_nature,nearest_school_level
0,serangoon,106,serangoon north avenue 1,serangoon garden secondary school,200.6,government school,co-ed school,secondary (s1-s5)
1,bukit merah,34,telok blangah way,chij st. theresa's convent,204.0,government-aided sch,girls' school,secondary (s1-s5)
2,punggol,226b,sumang lane,yusof ishak secondary school,558.0,government school,co-ed school,secondary (s1-s5)
3,marine parade,74,marine drive,chij (katong) primary,531.7,government-aided sch,girls' school,primary
4,choa chu kang,284,choa chu kang avenue 3,chua chu kang primary school,220.0,government school,co-ed school,primary
...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,chij secondary (toa payoh),260.3,government-aided sch,girls' school,secondary (s1-s5)
9460,sembawang,307,canberra road,sembawang primary school,120.3,government school,co-ed school,primary
9461,woodlands,681a,woodlands drive 62,greenwood primary school,110.9,government school,co-ed school,primary
9462,choa chu kang,673a,choa chu kang crescent,unity secondary school,200.6,government school,co-ed school,secondary (s1-s5)


In [86]:
mx =  pd.read_csv(f'{ROOT_PATH}/raw_model_data/collected_data/school_data/sg-schools-mixed.csv')
mx_sec = mx[['NAME', 'LATITUDE', 'LONGITUDE','pg3_min', 'pg3_max',]]
mx_jc = mx[['NAME', 'LATITUDE', 'LONGITUDE', 
       'l1r5_international_baccalaureate_min',
       'l1r5_international_baccalaureate_max',
       'l1r5_arts_min', 'l1r5_arts_max', 
       'l1r5_science_min', 'l1r5_science_max',]]
mx_jc['l1r5_arts_max'] = mx_jc['l1r5_arts_max'].fillna(mx_jc['l1r5_international_baccalaureate_max'])
mx_jc['l1r5_arts_min'] = mx_jc['l1r5_arts_min'].fillna(mx_jc['l1r5_international_baccalaureate_min'])
mx_jc['l1r5_science_max'] = mx_jc['l1r5_science_max'].fillna(mx_jc['l1r5_international_baccalaureate_max'])
mx_jc['l1r5_science_min'] = mx_jc['l1r5_science_min'].fillna(mx_jc['l1r5_international_baccalaureate_min'])
mx_jc = mx_jc.drop(['l1r5_international_baccalaureate_min','l1r5_international_baccalaureate_max'], axis = 1)
display(mx_sec)
display(mx_jc)

/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipykernel_28698/4194017020.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mx_jc['l1r5_arts_max'] = mx_jc['l1r5_arts_max'].fillna(mx_jc['l1r5_international_baccalaureate_max'])
/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipykernel_28698/4194017020.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mx_jc['l1r5_arts_min'] = mx_jc['l1r5_arts_min'].fillna(mx_jc['l1r5_international_baccalaureate_min'])
/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipy

,NAME,LATITUDE,LONGITUDE,pg3_min,pg3_max
0,anglo-chinese school (independent),1.303704,103.780363,7,13
1,catholic high school,1.354525,103.844901,6(D),8(M)
2,chij st. nicholas girls' school,1.373477,103.834253,4(D),7(M)
3,dunman high school,1.299962,103.882942,4(D),8(M)
4,hwa chong institution,1.324948,103.805704,4(D),6(M)
5,maris stella high school,1.341395,103.877852,7(M),12
6,national junior college,1.330293,103.804111,5,8
7,nus high school of mathematics and science,1.306007,103.770153,NaN,NaN
8,raffles institution,1.346188,103.845875,4,6
9,river valley high school,1.344078,103.709022,4(M),9(M)


,NAME,LATITUDE,LONGITUDE,l1r5_arts_min,l1r5_arts_max,l1r5_science_min,l1r5_science_max
0,anglo-chinese school (independent),1.303704,103.780363,2.0,5.0,2.0,5.0
1,catholic high school,1.354525,103.844901,NaN,NaN,NaN,NaN
2,chij st. nicholas girls' school,1.373477,103.834253,NaN,NaN,NaN,NaN
3,dunman high school,1.299962,103.882942,6.0,8.0,4.0,6.0
4,hwa chong institution,1.324948,103.805704,2.0,5.0,2.0,3.0
5,maris stella high school,1.341395,103.877852,NaN,NaN,NaN,NaN
6,national junior college,1.330293,103.804111,4.0,8.0,2.0,6.0
7,nus high school of mathematics and science,1.306007,103.770153,NaN,NaN,NaN,NaN
8,raffles institution,1.346188,103.845875,2.0,5.0,2.0,3.0
9,river valley high school,1.344078,103.709022,7.0,9.0,2.0,8.0


In [87]:
# Get nearest pri school
pri = pd.read_csv(f'{ROOT_PATH}/raw_model_data/collected_data/school_data/sg-schools-primary.csv')
pri = pri[['NAME', 'LATITUDE', 'LONGITUDE',]]
pri

,NAME,LATITUDE,LONGITUDE
0,admiralty primary school,1.442635,103.800040
1,ahmad ibrahim primary school,1.433153,103.832942
2,ai tong school,1.360583,103.833020
3,alexandra primary school,1.291334,103.824425
4,anchor green primary school,1.390370,103.887165
...,...,...,...
174,yuhua primary school,1.342802,103.741106
175,yumin primary school,1.351292,103.950462
176,zhangde primary school,1.284212,103.825952
177,zhenghua primary school,1.379549,103.769314


In [ ]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(pri.LATITUDE.values, pri.LONGITUDE.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

count_rows = []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_pri_school": pri.NAME[nearest],
        "nearest_pri_school_m": round(float(d[nearest]), 1),
    }

    count_rows.append(results)

pri_counts = pd.DataFrame(count_rows)
sch_type2 = sch_type.copy(deep=True)
sch_type2.columns = ['nearest_pri_school', 'pri_sch_type', 'pri_sch_nature']
pri_counts = pd.merge(pri_counts, sch_type2, how = 'left', on = ['nearest_pri_school'])
pri_counts

,TOWN,BLOCK,STREET,nearest_pri_school,nearest_pri_school_m,pri_sch_type,pri_sch_nature
0,serangoon,106,serangoon north avenue 1,rosyth school,549.8,government school,co-ed school
1,bukit merah,34,telok blangah way,radin mas primary school,248.3,government school,co-ed school
2,punggol,226b,sumang lane,punggol green primary school,584.2,government school,co-ed school
3,marine parade,74,marine drive,chij (katong) primary,531.7,government-aided sch,girls' school
4,choa chu kang,284,choa chu kang avenue 3,chua chu kang primary school,220.0,government school,co-ed school
...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,chij primary (toa payoh),416.0,government-aided sch,girls' school
9460,sembawang,307,canberra road,sembawang primary school,120.3,government school,co-ed school
9461,woodlands,681a,woodlands drive 62,greenwood primary school,110.9,government school,co-ed school
9462,choa chu kang,673a,choa chu kang crescent,unity primary school,219.0,government school,co-ed school


In [91]:
sec =  pd.read_csv(f'{ROOT_PATH}/raw_model_data/collected_data/school_data/sg-schools-secondary.csv')
sec = sec[['NAME', 'LATITUDE', 'LONGITUDE','pg3_min', 'pg3_max',]]
sec = pd.concat([sec, mx_sec], axis = 0).reset_index(drop = True)
sec

,NAME,LATITUDE,LONGITUDE,pg3_min,pg3_max
0,admiralty secondary school,1.445891,103.802398,16,22
1,ahmad ibrahim secondary school,1.436060,103.829719,11,18
2,anderson secondary school,1.374340,103.851554,7,11
3,ang mo kio secondary school,1.367337,103.842155,12,16
4,anglican high school,1.331181,103.941853,4,12
...,...,...,...,...,...
142,river valley high school,1.344078,103.709022,4(M),9(M)
143,"school of the arts, singapore",1.299577,103.848762,NaN,NaN
144,singapore sports school,1.426411,103.789445,NaN,NaN
145,st. joseph's institution,1.297258,103.851024,4,8


In [92]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(sec.LATITUDE.values, sec.LONGITUDE.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

count_rows = []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_sec_school": sec.NAME[nearest],
        "nearest_sec_school_m": round(float(d[nearest]), 1),
    }

    count_rows.append(results)
sec2 = sec[['NAME','pg3_min','pg3_max']]
sec2.columns = ['nearest_sec_school','sec_sch_score_entry_min','sec_sch_score_entry_max']
sec_counts = pd.DataFrame(count_rows)
sec_counts = pd.merge(sec_counts, sec2, how = 'left',on = 'nearest_sec_school')
sec_counts['sec_sch_score_entry_max'] = sec_counts['sec_sch_score_entry_max'].fillna('-1').apply(lambda x: int(x.split('(')[0]))
sec_counts['sec_sch_score_entry_min'] = sec_counts['sec_sch_score_entry_min'].fillna('-1').apply(lambda x: int(x.split('(')[0]))
sch_type2.columns = ['nearest_sec_school', 'sec_sch_type', 'sec_sch_nature']
sec_counts = pd.merge(sec_counts, sch_type2, how = 'left', on = ['nearest_sec_school'])

sec_counts

,TOWN,BLOCK,STREET,nearest_sec_school,nearest_sec_school_m,sec_sch_score_entry_min,sec_sch_score_entry_max,sec_sch_type,sec_sch_nature
0,serangoon,106,serangoon north avenue 1,serangoon garden secondary school,200.6,12,22,government school,co-ed school
1,bukit merah,34,telok blangah way,chij st. theresa's convent,204.0,9,20,government-aided sch,girls' school
2,punggol,226b,sumang lane,yusof ishak secondary school,558.0,7,15,government school,co-ed school
3,marine parade,74,marine drive,chij katong convent,1145.1,12,20,government-aided sch,girls' school
4,choa chu kang,284,choa chu kang avenue 3,bukit panjang govt. high school,505.3,7,11,government school,co-ed school
...,...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,chij secondary (toa payoh),260.3,8,20,government-aided sch,girls' school
9460,sembawang,307,canberra road,sembawang secondary school,539.4,18,22,government school,co-ed school
9461,woodlands,681a,woodlands drive 62,spectra secondary school,279.6,-1,-1,specialised independent school,co-ed school
9462,choa chu kang,673a,choa chu kang crescent,unity secondary school,200.6,12,18,government school,co-ed school


In [93]:
jc = pd.read_csv(f'{ROOT_PATH}/raw_model_data/collected_data/school_data/sg-schools-junior_college.csv')
jc = jc[[
'NAME', 'LATITUDE', 'LONGITUDE', 'l1r5_arts_min', 'l1r5_arts_max', 
'l1r5_science_min', 'l1r5_science_max',
]]
jc = pd.concat([jc, mx_jc], axis = 0).reset_index(drop = True)
for col in ['l1r5_arts_min', 'l1r5_arts_max', 
'l1r5_science_min', 'l1r5_science_max',]:
    jc[col] = jc[col].fillna(-1).apply(lambda x: int(x))
jc

,NAME,LATITUDE,LONGITUDE,l1r5_arts_min,l1r5_arts_max,l1r5_science_min,l1r5_science_max
0,anderson serangoon junior college,1.365043,103.887039,7,10,3,9
1,anglo-chinese junior college,1.303043,103.785358,4,9,2,7
2,catholic junior college,1.324556,103.835029,5,12,6,11
3,eunoia junior college,1.362138,103.841286,2,6,2,5
4,jurong pioneer junior college,1.384374,103.754375,9,14,4,14
5,millennia institute,1.350319,103.741594,10,20,9,20
6,nanyang junior college,1.350283,103.866499,2,7,2,5
7,st andrew's junior college,1.333065,103.863576,5,10,3,8
8,tampines meridian junior college,1.376333,103.936991,7,12,7,12
9,victoria junior college,1.306702,103.919983,4,8,2,6


In [94]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(jc.LATITUDE.values, jc.LONGITUDE.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

count_rows = []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_jc_school": jc.NAME[nearest],
        "nearest_jc_school_m": round(float(d[nearest]), 1),
    }

    count_rows.append(results)
jc2 = jc[['NAME', 'l1r5_arts_min', 'l1r5_arts_max', 'l1r5_science_min', 'l1r5_science_max']]
jc2.columns = ['nearest_jc_school','jc_arts_entry_min','jc_arts_entry_max','jc_science_entry_min','jc_science_entry_max']
jc_counts = pd.DataFrame(count_rows)
jc_counts = pd.merge(jc_counts, jc2, how = 'left', on = 'nearest_jc_school')
sch_type2.columns = ['nearest_jc_school', 'jc_sch_type', 'jc_sch_nature']
jc_counts = pd.merge(jc_counts, sch_type2, how = 'left', on = ['nearest_jc_school'])
jc_counts

,TOWN,BLOCK,STREET,nearest_jc_school,nearest_jc_school_m,jc_arts_entry_min,jc_arts_entry_max,jc_science_entry_min,jc_science_entry_max,jc_sch_type,jc_sch_nature
0,serangoon,106,serangoon north avenue 1,anderson serangoon junior college,1893.1,7,10,3,9,government school,co-ed school
1,bukit merah,34,telok blangah way,"school of the arts, singapore",4114.8,-1,-1,-1,-1,specialised school,co-ed school
2,punggol,226b,sumang lane,anderson serangoon junior college,4248.8,7,10,3,9,government school,co-ed school
3,marine parade,74,marine drive,victoria junior college,1458.0,4,8,2,6,government school,co-ed school
4,choa chu kang,284,choa chu kang avenue 3,jurong pioneer junior college,1455.6,9,14,4,14,government school,co-ed school
...,...,...,...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,catholic junior college,1489.1,5,12,6,11,government-aided sch,co-ed school
9460,sembawang,307,canberra road,yishun innova junior college,2581.7,5,20,9,18,government school,co-ed school
9461,woodlands,681a,woodlands drive 62,singapore sports school,2162.1,-1,-1,-1,-1,specialised school,co-ed school
9462,choa chu kang,673a,choa chu kang crescent,jurong pioneer junior college,2000.9,9,14,4,14,government school,co-ed school


In [95]:
df_lst = [pri_counts, sec_counts, jc_counts]
sch_counts = counts
for df in df_lst:
    sch_counts = pd.merge(sch_counts, df, how = 'left', on = ['TOWN',"BLOCK", 'STREET'])
sch_counts

,TOWN,BLOCK,STREET,nearest_school,nearest_school_m,nearest_school_type,nearest_school_nature,nearest_school_level,nearest_pri_school,nearest_pri_school_m,...,sec_sch_type,sec_sch_nature,nearest_jc_school,nearest_jc_school_m,jc_arts_entry_min,jc_arts_entry_max,jc_science_entry_min,jc_science_entry_max,jc_sch_type,jc_sch_nature
0,serangoon,106,serangoon north avenue 1,serangoon garden secondary school,200.6,government school,co-ed school,secondary (s1-s5),rosyth school,549.8,...,government school,co-ed school,anderson serangoon junior college,1893.1,7,10,3,9,government school,co-ed school
1,bukit merah,34,telok blangah way,chij st. theresa's convent,204.0,government-aided sch,girls' school,secondary (s1-s5),radin mas primary school,248.3,...,government-aided sch,girls' school,"school of the arts, singapore",4114.8,-1,-1,-1,-1,specialised school,co-ed school
2,punggol,226b,sumang lane,yusof ishak secondary school,558.0,government school,co-ed school,secondary (s1-s5),punggol green primary school,584.2,...,government school,co-ed school,anderson serangoon junior college,4248.8,7,10,3,9,government school,co-ed school
3,marine parade,74,marine drive,chij (katong) primary,531.7,government-aided sch,girls' school,primary,chij (katong) primary,531.7,...,government-aided sch,girls' school,victoria junior college,1458.0,4,8,2,6,government school,co-ed school
4,choa chu kang,284,choa chu kang avenue 3,chua chu kang primary school,220.0,government school,co-ed school,primary,chua chu kang primary school,220.0,...,government school,co-ed school,jurong pioneer junior college,1455.6,9,14,4,14,government school,co-ed school
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,chij secondary (toa payoh),260.3,government-aided sch,girls' school,secondary (s1-s5),chij primary (toa payoh),416.0,...,government-aided sch,girls' school,catholic junior college,1489.1,5,12,6,11,government-aided sch,co-ed school
9460,sembawang,307,canberra road,sembawang primary school,120.3,government school,co-ed school,primary,sembawang primary school,120.3,...,government school,co-ed school,yishun innova junior college,2581.7,5,20,9,18,government school,co-ed school
9461,woodlands,681a,woodlands drive 62,greenwood primary school,110.9,government school,co-ed school,primary,greenwood primary school,110.9,...,specialised independent school,co-ed school,singapore sports school,2162.1,-1,-1,-1,-1,specialised school,co-ed school
9462,choa chu kang,673a,choa chu kang crescent,unity secondary school,200.6,government school,co-ed school,secondary (s1-s5),unity primary school,219.0,...,government school,co-ed school,jurong pioneer junior college,2000.9,9,14,4,14,government school,co-ed school


In [96]:
sch_counts.columns

Index(['TOWN', 'BLOCK', 'STREET', 'nearest_school', 'nearest_school_m',
       'nearest_school_type', 'nearest_school_nature', 'nearest_school_level',
       'nearest_pri_school', 'nearest_pri_school_m', 'pri_sch_type',
       'pri_sch_nature', 'nearest_sec_school', 'nearest_sec_school_m',
       'sec_sch_score_entry_min', 'sec_sch_score_entry_max', 'sec_sch_type',
       'sec_sch_nature', 'nearest_jc_school', 'nearest_jc_school_m',
       'jc_arts_entry_min', 'jc_arts_entry_max', 'jc_science_entry_min',
       'jc_science_entry_max', 'jc_sch_type', 'jc_sch_nature'],
      dtype='object')

In [98]:
sch_counts[['TOWN', 'BLOCK', 'STREET', 'nearest_school', 'nearest_school_m',
       'nearest_school_type', 'nearest_school_nature', 'nearest_school_level',
       'nearest_pri_school', 'nearest_pri_school_m', 'pri_sch_type',
       'pri_sch_nature', 'nearest_sec_school', 'nearest_sec_school_m',
       'sec_sch_score_entry_min', 'sec_sch_score_entry_max', 'sec_sch_type',
       'sec_sch_nature', 'nearest_jc_school', 'nearest_jc_school_m',
       'jc_arts_entry_min', 'jc_arts_entry_max', 'jc_science_entry_min',
       'jc_science_entry_max', 'jc_sch_type', 'jc_sch_nature']].to_csv(
           f'{ROOT_PATH}/graph_feature_store/hdb_nearest_school.csv', index = False
       )

# Join all features

In [99]:
# Keys, train and test set
train_house = pd.read_csv(f'{ROOT_PATH}/graph_feature_store/train-house.csv')
test_house = pd.read_csv(f'{ROOT_PATH}/graph_feature_store/test-house.csv')
stat_key = pd.read_csv(f'{ROOT_PATH}/graph_feature_store/station-keys.csv')[[
    'NAME', 'nearest_mall', 'n_lines', 'status',
       'n_malls_500m', 'nearest_mall_m'
]]
stat_key.columns = ['nearest_station','nearest_mall','train_lines','status','n_malls_500m','nearest_mall_m']
stat_key['status'] = stat_key['status'].apply(lambda x: 1 if x == 'open' else 0)
stat_key.columns = ['nearest_station','nearest_mall','train_lines','station_is_open','n_malls_500m','nearest_mall_m']
closest_mrt_station = pd.read_csv(f'{ROOT_PATH}/graph_feature_store/hdb_nearest_station.csv')
closest_school = pd.read_csv(f'{ROOT_PATH}/graph_feature_store/hdb_nearest_school.csv')

# Train Traffic Features
traffic = pd.read_csv(f'{ROOT_PATH}/graph_feature_store/station-traffic-std.csv')
traffic.columns = ['nearest_station', 'mrt_traffic_tap_in_amount_norm',
       'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm',
       'mrt_traffic_weekday_tap_norm', 'mrt_traffic_weekend_tap_norm',
       'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm',
       'mrt_traffic_busiest_hour']

# Cluster Output
train_lap = pd.read_csv(f'{ROOT_PATH}/graph_feature_store/train_cluster.csv').drop(['LATITUDE','LONGITUDE','POSTAL_CODE'], axis = 1)
test_lap = pd.read_csv(f'{ROOT_PATH}/graph_feature_store/test_cluster.csv').drop(['LATITUDE','LONGITUDE','POSTAL_CODE'], axis = 1)

# Planning Area features
plan_area = pd.read_csv(f'{ROOT_PATH}/raw_model_data/collected_data/zoning_data/sg-demographics-planning-area.csv')[[
    'planning_area',
    'total_population', 
    'pct_age_0_14', 
    'pct_age_15_64', 
    'pct_age_65_plus',
    'unemployment_rate', 
    'total_households', 
    'pct_hdb',
    'pct_private_housing', 'pct_hh_income_ge_10k', 'pct_hh_income_below_2k',
    'avg_household_size', 'pct_chinese', 'pct_malay', 'pct_indian',
    'pct_owner_occupied', 'pct_pmet'
]]
plan_ft = []
for c in plan_area.columns:
    if c != 'planning_area':
        plan_ft.append(f'plan_area_ft_{c}')
    else:
        plan_ft.append('PLANNING_AREA')
plan_area.columns = plan_ft

# Subzone Area features
subzone = pd.read_csv(f'{ROOT_PATH}/raw_model_data/collected_data/zoning_data/sg-demographics-subzone.csv')[
    ['total_population', 'pct_age_0_14', 'pct_age_15_64', 'pct_age_65_plus', 'pct_male']
]

# Mall Features
mall = pd.read_csv(f'{ROOT_PATH}/raw_model_data/collected_data/mall_features/sg-mall-features-wikipedia.csv')


In [100]:
closest_school

,TOWN,BLOCK,STREET,nearest_school,nearest_school_m,nearest_school_type,nearest_school_nature,nearest_school_level,nearest_pri_school,nearest_pri_school_m,...,sec_sch_type,sec_sch_nature,nearest_jc_school,nearest_jc_school_m,jc_arts_entry_min,jc_arts_entry_max,jc_science_entry_min,jc_science_entry_max,jc_sch_type,jc_sch_nature
0,serangoon,106,serangoon north avenue 1,serangoon garden secondary school,200.6,government school,co-ed school,secondary (s1-s5),rosyth school,549.8,...,government school,co-ed school,anderson serangoon junior college,1893.1,7,10,3,9,government school,co-ed school
1,bukit merah,34,telok blangah way,chij st. theresa's convent,204.0,government-aided sch,girls' school,secondary (s1-s5),radin mas primary school,248.3,...,government-aided sch,girls' school,"school of the arts, singapore",4114.8,-1,-1,-1,-1,specialised school,co-ed school
2,punggol,226b,sumang lane,yusof ishak secondary school,558.0,government school,co-ed school,secondary (s1-s5),punggol green primary school,584.2,...,government school,co-ed school,anderson serangoon junior college,4248.8,7,10,3,9,government school,co-ed school
3,marine parade,74,marine drive,chij (katong) primary,531.7,government-aided sch,girls' school,primary,chij (katong) primary,531.7,...,government-aided sch,girls' school,victoria junior college,1458.0,4,8,2,6,government school,co-ed school
4,choa chu kang,284,choa chu kang avenue 3,chua chu kang primary school,220.0,government school,co-ed school,primary,chua chu kang primary school,220.0,...,government school,co-ed school,jurong pioneer junior college,1455.6,9,14,4,14,government school,co-ed school
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,chij secondary (toa payoh),260.3,government-aided sch,girls' school,secondary (s1-s5),chij primary (toa payoh),416.0,...,government-aided sch,girls' school,catholic junior college,1489.1,5,12,6,11,government-aided sch,co-ed school
9460,sembawang,307,canberra road,sembawang primary school,120.3,government school,co-ed school,primary,sembawang primary school,120.3,...,government school,co-ed school,yishun innova junior college,2581.7,5,20,9,18,government school,co-ed school
9461,woodlands,681a,woodlands drive 62,greenwood primary school,110.9,government school,co-ed school,primary,greenwood primary school,110.9,...,specialised independent school,co-ed school,singapore sports school,2162.1,-1,-1,-1,-1,specialised school,co-ed school
9462,choa chu kang,673a,choa chu kang crescent,unity secondary school,200.6,government school,co-ed school,secondary (s1-s5),unity primary school,219.0,...,government school,co-ed school,jurong pioneer junior college,2000.9,9,14,4,14,government school,co-ed school


In [101]:
# Join Cluster
print(f'Train: {train_house.shape}')
print(f'Test: {test_house.shape}')
train = pd.merge(train_house, train_lap, how = 'left',on = ['TOWN', 'BLOCK','STREET',]).reset_index(drop=True)
test = pd.merge(test_house, test_lap, how = 'left',on = ['TOWN', 'BLOCK','STREET',]).drop_duplicates().reset_index(drop=True)

# Join Planning Area
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, plan_area, how = 'left',on = ['PLANNING_AREA'])
test = pd.merge(test, plan_area, how = 'left',on = ['PLANNING_AREA'])

# Join nearest_station
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, closest_mrt_station, how = 'left',on = ['TOWN', 'BLOCK','STREET'])
test = pd.merge(test, closest_mrt_station, how = 'left',on = ['TOWN', 'BLOCK','STREET'])

# Join nearest_school
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, closest_school, how = 'left',on = ['TOWN', 'BLOCK','STREET'])
test = pd.merge(test, closest_school, how = 'left',on = ['TOWN', 'BLOCK','STREET'])

# Join MRT Traffic
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, traffic, how = 'left',on = ['nearest_station'])
test = pd.merge(test, traffic, how = 'left',on = ['nearest_station'])

# Join Nearest Mall
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, stat_key, how = 'left',on = ['nearest_station'])
test = pd.merge(test, stat_key, how = 'left',on = ['nearest_station'])

print(f'Train: {train.shape}')
print(f'Test: {test.shape}')

Train: (149359, 18)
Test: (42236, 17)
Train: (149359, 20)
Test: (42236, 19)
Train: (149359, 36)
Test: (42236, 35)
Train: (149359, 40)
Test: (42236, 39)
Train: (149359, 63)
Test: (42236, 62)
Train: (149359, 71)
Test: (42236, 70)
Train: (149359, 76)
Test: (42236, 75)


In [ ]:
'nearest_school_type', 'nearest_school_nature', 'nearest_school_level','pri_sch_type',
'pri_sch_nature',


 'sec_sch_type',
'sec_sch_nature',  'jc_sch_type', 'jc_sch_nature',

In [103]:
label = 'MONTHLY_RENT'

id_col = ['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH', 'LEASE_COMMENCE_DATE',
       'TOWN', 'BLOCK', 'STREET',
       'POSTAL_CODE', 'LATITUDE', 'LONGITUDE', 'YEAR_COMPLETED',
       'SUBZONE', 'PLANNING_AREA', 'REGION', 'nearest_station', 'nearest_mall',
       'nearest_school','nearest_pri_school', 'nearest_sec_school', 'nearest_jc_school', 
       ]

num_col = ['FLOOR_AREA_SQM','plan_area_ft_total_population',
       'plan_area_ft_pct_age_0_14', 'plan_area_ft_pct_age_15_64',
       'plan_area_ft_pct_age_65_plus', 'plan_area_ft_unemployment_rate',
       'plan_area_ft_total_households', 'plan_area_ft_pct_hdb',
       'plan_area_ft_pct_private_housing', 'plan_area_ft_pct_hh_income_ge_10k',
       'plan_area_ft_pct_hh_income_below_2k',
       'plan_area_ft_avg_household_size', 'plan_area_ft_pct_chinese',
       'plan_area_ft_pct_malay', 'plan_area_ft_pct_indian',
       'plan_area_ft_pct_owner_occupied', 'plan_area_ft_pct_pmet',
       'nearest_station_m', 'n_stations_500m',
       'n_stations_1000m', 'mrt_traffic_tap_in_amount_norm','n_malls_500m', 'nearest_mall_m',
       'nearest_school_m','nearest_pri_school_m', 
       'nearest_sec_school_m',
       'sec_sch_score_entry_min', 'sec_sch_score_entry_max',
       'nearest_jc_school_m',
       'jc_arts_entry_min', 'jc_arts_entry_max', 'jc_science_entry_min',
       'jc_science_entry_max',
       'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm',
       'mrt_traffic_weekday_tap_norm', 'mrt_traffic_weekend_tap_norm',
       'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm']

cat_col_num = ['MAX_FLOOR',
 'dbscan_cluster',
 'diffusion_cluster',
 'mrt_traffic_busiest_hour',
 'train_lines',
 'station_is_open']
cat_col = [i for i in test.columns if i not in num_col+id_col+[label]+cat_col_num]

In [111]:
cat_col

['FLAT_TYPE',
 'FLAT_MODEL',
 'nearest_school_type',
 'nearest_school_nature',
 'nearest_school_level',
 'pri_sch_type',
 'pri_sch_nature',
 'sec_sch_type',
 'sec_sch_nature',
 'jc_sch_type',
 'jc_sch_nature']

In [106]:
# Numerical Columns
## Handle Null values
for n in num_col:
    train[n] = train[n].fillna(-1)
    test[n] = test[n].fillna(-1)    

## Standard Scaler
num_scale = StandardScaler()
train[num_col] = num_scale.fit_transform(train[num_col])
test[num_col] = num_scale.transform(test[num_col])
joblib.dump(num_scale, f'{ROOT_PATH}/scalers/numerical_scale.pkl')


['/Users/keltonn/Desktop/Projects/Data/location_data_mining_cs5228/00_location_graph_data_mining/scalers/numerical_scale.pkl']

In [107]:
# Categorical Columns
for c in cat_col_num:
    train[c] = train[c].fillna(-1).apply(lambda x: int(x))
    test[c] = test[c].fillna(-1).apply(lambda x: int(x))


In [110]:
train.to_csv(f'{ROOT_PATH}/train_feature.csv',index = False)
test.to_csv(f'{ROOT_PATH}/test_feature.csv',index = False)

In [ ]:

# # One hot encode strings
# enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# enc.set_output(transform="pandas")

# trt = enc.fit_transform(train[cat_col])
# tst = enc.fit_transform(test[cat_col])
# train = pd.concat([train,trt], axis = 1)
# test = pd.concat([test,tst], axis = 1)
# one_hot_col = ['FLAT_TYPE_1 room',
#        'FLAT_TYPE_2 room', 'FLAT_TYPE_3 room', 'FLAT_TYPE_4 room',
#        'FLAT_TYPE_5 room', 'FLAT_TYPE_executive', 'FLAT_MODEL_2 room',
#        'FLAT_MODEL_3gen', 'FLAT_MODEL_adjoined flat', 'FLAT_MODEL_apartment',
#        'FLAT_MODEL_dbss', 'FLAT_MODEL_improved',
#        'FLAT_MODEL_improved maisonette', 'FLAT_MODEL_maisonette',
#        'FLAT_MODEL_model a', 'FLAT_MODEL_model a maisonette',
#        'FLAT_MODEL_model a2', 'FLAT_MODEL_new generation',
#        'FLAT_MODEL_premium apartment', 'FLAT_MODEL_premium apartment loft',
#        'FLAT_MODEL_premium maisonette', 'FLAT_MODEL_simplified',
#        'FLAT_MODEL_standard', 'FLAT_MODEL_terrace', 'FLAT_MODEL_type s1',
#        'FLAT_MODEL_type s2']

# for c in one_hot_col:
#     if c not in test.columns:
#         test[c] = 0
# final_train = train[id_col+num_col+cat_col_num+cat_col + one_hot_col + [label]]
# final_test = test[id_col+num_col+cat_col_num+cat_col+one_hot_col]